# NIA T32 Landscape Analysis

This notebook queries NIH Reporter for T32 grant information and publications, prepares input files for VOSviewer bibliometric mapping, and produces charts summarizing the research landscape and measurement science gaps.

---

## Workflow overview

| Section | What it does | Requires Claude? |
|---|---|---|
| 1. Setup | Imports, paths, and folder creation | No |
| 2. NIH Reporter query | Fetches grant info and publications | No |
| 3. VOSviewer corpus | Builds grant abstract corpus for VOSviewer | No |
| 4. MEDLINE export | Fetches publication records from PubMed | No |
| 5. VOSviewer map analysis | Identifies measurement terms, generates charts | **Yes — see cell instructions** |

---

## Output folder structure

```
output/
├── nih_reporter/          ← NIH Reporter & PubMed data files
├── vosviewer_input/       ← files you feed INTO VOSviewer
├── vosviewer_output/      ← map and network files VOSviewer exports
├── vosviewer_images/      ← VOSviewer screenshot exports (added manually)
├── charts/                ← Python-generated charts
└── measurement_analysis/  ← measurement terms and excluded terms CSVs
```

## Section 1 — Setup

In [ ]:
# ── Install dependencies if needed ─────────────────────────────────────────────
# !pip install requests pandas openpyxl tqdm matplotlib

import requests
import pandas as pd
import matplotlib.pyplot as plt
import re
import csv
import time
from tqdm.notebook import tqdm
from pathlib import Path
import os

print(f"Working directory : {os.getcwd()}")
print(f"Contents          : {os.listdir()}")

In [ ]:
# ── USER SETTINGS — edit these paths to match your project folder ──────────────
CSV_PATH  = "data/grants.csv"              # Input CSV with grant numbers
GRANT_COL = "Project Number"               # Column name containing grant numbers
EMAIL     = "your_email@institution.edu"   # Used by PubMed API to identify you
# ──────────────────────────────────────────────────────────────────────────────

# ── Output folder structure — all subfolders created automatically ─────────────
OUT_DIR          = Path("output")
DIR_NIH          = OUT_DIR / "nih_reporter"         # NIH Reporter & PubMed data
DIR_VOS_INPUT    = OUT_DIR / "vosviewer_input"      # Files fed INTO VOSviewer
DIR_VOS_OUTPUT   = OUT_DIR / "vosviewer_output"     # Files VOSviewer exports
DIR_VOS_IMAGES   = OUT_DIR / "vosviewer_images"     # VOSviewer screenshots (manual)
DIR_CHARTS       = OUT_DIR / "charts"               # Python-generated charts
DIR_MEAS         = OUT_DIR / "measurement_analysis" # Measurement & excluded terms

for folder in [OUT_DIR, DIR_NIH, DIR_VOS_INPUT, DIR_VOS_OUTPUT,
               DIR_VOS_IMAGES, DIR_CHARTS, DIR_MEAS]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"  {'Created' if not folder.exists() else 'Ready  '} : {folder}")

# ── Derived file paths — do not edit ──────────────────────────────────────────
OUTPUT_FILE    = DIR_NIH / "nih_results.xlsx"
PMID_DOI_FILE  = DIR_NIH / "grant_pmid_doi.csv"
MEDLINE_FILE   = DIR_VOS_INPUT / "grant_publications_medline.txt"
CORPUS_FILE    = DIR_VOS_INPUT / "vosviewer_corpus.txt"
TITLES_FILE    = DIR_VOS_INPUT / "vosviewer_titles.txt"
THESAURUS_FILE = DIR_VOS_INPUT / "vosviewer_thesaurus.txt"

# ── API constants — do not edit ────────────────────────────────────────────────
BASE_URL      = "https://api.reporter.nih.gov/v2"
HEADERS       = {"Content-Type": "application/json", "accept": "application/json"}
PAGE_SIZE     = 500
DELAY_SECONDS = 0.5

print("\nAll folders ready. File paths set.")

## Section 2 — NIH Reporter Query

In [ ]:
# Load grant numbers from CSV
df_input = pd.read_csv(CSV_PATH)
print(f"Columns found : {df_input.columns.tolist()}")
print(f"Rows          : {len(df_input)}")
df_input.head()

In [ ]:
# Clean and deduplicate grant numbers
grant_numbers = (
    df_input[GRANT_COL]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .drop_duplicates()
    .tolist()
)
print(f"Unique grant numbers to query: {len(grant_numbers)}")
print(grant_numbers[:5])

In [ ]:
# API helper functions

def fetch_grant_info(grant_number: str) -> list[dict]:
    """Query NIH Reporter for a single grant number, paginating across all fiscal years."""
    url = f"{BASE_URL}/projects/search"
    all_records = []
    offset = 0
    while True:
        payload = {
            "criteria": {"project_nums": [grant_number]},
            "offset": offset,
            "limit": PAGE_SIZE,
            "fields": [
                "project_num", "project_title", "fiscal_year",
                "project_start_date", "project_end_date", "award_amount",
                "agency_ic_admin", "organization", "principal_investigators",
                "program_officers", "abstract_text", "activity_code",
                "award_type", "full_study_section", "is_active", "appl_id"
            ]
        }
        try:
            resp = requests.post(url, headers=HEADERS, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  WARNING: Error fetching grant {grant_number}: {e}")
            break
        page_results = data.get("results", [])
        all_records.extend(page_results)
        total = data.get("total")
        offset += len(page_results)
        if not page_results:
            break
        if total is not None and offset >= total:
            break
        if len(page_results) < PAGE_SIZE:
            break
    return all_records


def fetch_publications(appl_id: int, grant_number: str = "") -> list[dict]:
    """Fetch all publications for one appl_id, paging until a short/empty page."""
    url = f"{BASE_URL}/publications/search"
    all_pubs = []
    offset = 0
    page_num = 0
    while True:
        payload = {
            "criteria": {"appl_ids": [appl_id]},
            "offset": offset,
            "limit": PAGE_SIZE
        }
        try:
            resp = requests.post(url, headers=HEADERS, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  WARNING: Error fetching publications for appl_id {appl_id}: {e}")
            break
        page_results = data.get("results", [])
        all_pubs.extend(page_results)
        page_num += 1
        offset += len(page_results)
        if page_num > 1:
            print(f"    {grant_number} appl_id {appl_id}: fetched {offset} publications so far...")
        if len(page_results) < PAGE_SIZE:
            break
        time.sleep(DELAY_SECONDS)
    return all_pubs


def flatten_grant(record: dict, queried_grant_num: str) -> dict:
    org = record.get("organization") or {}
    pis = record.get("principal_investigators") or []
    pi_names = "; ".join(
        f"{p.get('last_name', '')}, {p.get('first_name', '')}".strip(", ") for p in pis
    )
    return {
        "queried_grant_number":    queried_grant_num,
        "project_num":             record.get("project_num"),
        "appl_id":                 record.get("appl_id"),
        "project_title":           record.get("project_title"),
        "fiscal_year":             record.get("fiscal_year"),
        "project_start_date":      record.get("project_start_date"),
        "project_end_date":        record.get("project_end_date"),
        "is_active":               record.get("is_active"),
        "award_amount":            record.get("award_amount"),
        "activity_code":           record.get("activity_code"),
        "award_type":              record.get("award_type"),
        "agency_ic_admin":         (record.get("agency_ic_admin") or {}).get("abbreviation"),
        "organization_name":       org.get("org_name"),
        "organization_city":       org.get("org_city"),
        "organization_state":      org.get("org_state"),
        "principal_investigators": pi_names,
        "study_section":           (record.get("full_study_section") or {}).get("name"),
        "abstract_text":           record.get("abstract_text"),
    }


def flatten_publication(pub: dict, queried_grant_num: str, appl_id: int) -> dict:
    return {
        "queried_grant_number": queried_grant_num,
        "appl_id":              appl_id,
        "pmid":                 pub.get("pmid"),
        "title":                pub.get("title"),
        "authors":              pub.get("author_list"),
        "journal":              pub.get("journal_title") or pub.get("journal_title_abbreviation"),
        "pub_date":             pub.get("pub_date"),
        "pub_year":             pub.get("pub_year"),
        "doi":                  pub.get("doi"),
        "pubmed_url":           f"https://pubmed.ncbi.nlm.nih.gov/{pub.get('pmid')}/" if pub.get("pmid") else None,
    }

print("Functions defined.")

In [ ]:
# Run the main query loop
all_grants = []
all_publications = []
not_found = []

for grant_num in tqdm(grant_numbers, desc="Querying NIH Reporter"):
    records = fetch_grant_info(grant_num)
    if not records:
        not_found.append(grant_num)
        time.sleep(DELAY_SECONDS)
        continue
    appl_ids_seen = set()
    for record in records:
        all_grants.append(flatten_grant(record, grant_num))
        appl_id = record.get("appl_id")
        if appl_id and appl_id not in appl_ids_seen:
            appl_ids_seen.add(appl_id)
            time.sleep(DELAY_SECONDS)
            pubs = fetch_publications(appl_id, grant_number=grant_num)
            for pub in pubs:
                all_publications.append(flatten_publication(pub, grant_num, appl_id))
    time.sleep(DELAY_SECONDS)

print(f"\nDone!")
print(f"  Grant records      : {len(all_grants):,}")
print(f"  Publication records: {len(all_publications):,}")
if not_found:
    print(f"  Not found ({len(not_found)}): {not_found}")

In [ ]:
# Preview and save results → output/nih_reporter/nih_results.xlsx
df_grants = pd.DataFrame(all_grants)
df_pubs   = pd.DataFrame(all_publications)

display(df_grants.head())
display(df_pubs.head())

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df_grants.to_excel(writer, sheet_name="Grant Info",   index=False)
    df_pubs.to_excel(  writer, sheet_name="Publications", index=False)
    if not_found:
        pd.DataFrame({"grant_number_not_found": not_found}).to_excel(
            writer, sheet_name="Not Found", index=False
        )

print(f"Saved: {OUTPUT_FILE.resolve()}")

In [ ]:
# Build PMID → DOI mapping and save → output/nih_reporter/grant_pmid_doi.csv
BATCH_SIZE = 200
DELAY      = 0.4

pmids = (
    df_pubs["pmid"]
    .dropna().astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .loc[lambda s: s != ""]
    .drop_duplicates().tolist()
)
print(f"Unique PMIDs to convert: {len(pmids):,}")


def fetch_dois_for_pmids(pmid_batch: list[str]) -> dict[str, str]:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    params = {"db": "pubmed", "id": ",".join(pmid_batch), "retmode": "json"}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"  WARNING: {e}")
        return {}
    result = {}
    for pmid, record in data.get("result", {}).items():
        if pmid == "uids":
            continue
        for id_obj in record.get("articleids", []):
            if id_obj.get("idtype") == "doi":
                doi = id_obj.get("value", "").strip()
                if doi:
                    result[pmid] = doi
                break
    return result


pmid_to_doi = {}
batches = [pmids[i:i+BATCH_SIZE] for i in range(0, len(pmids), BATCH_SIZE)]
for i, batch in enumerate(batches):
    print(f"  Batch {i+1}/{len(batches)}  ({len(pmid_to_doi):,} DOIs found so far)...")
    pmid_to_doi.update(fetch_dois_for_pmids(batch))
    time.sleep(DELAY)

print(f"\nPMIDs with DOI    : {len(pmid_to_doi):,}")
print(f"PMIDs without DOI : {len(pmids) - len(pmid_to_doi):,}")

# Build full table: grant number + PMID + DOI
df_mapped = df_pubs.copy()
df_mapped["pmid_clean"] = (
    df_mapped["pmid"].astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
)
df_mapped["doi"] = df_mapped["pmid_clean"].map(pmid_to_doi).fillna("no doi")

full_table = (
    df_mapped[["queried_grant_number", "pmid_clean", "doi"]]
    .rename(columns={"pmid_clean": "pmid"})
    .drop_duplicates()
    .sort_values(["queried_grant_number", "pmid"])
)
full_table.to_csv(PMID_DOI_FILE, index=False)
print(f"Saved: {PMID_DOI_FILE.resolve()}")

In [ ]:
# Summary statistics
print("Grant counts by fiscal year:")
display(df_grants["fiscal_year"].value_counts().sort_index())

if "award_amount" in df_grants.columns:
    print(f"\nTotal award amount: ${df_grants['award_amount'].sum():,.0f}")

print("\nPublications per grant (top 10):")
display(
    df_pubs.groupby("queried_grant_number")["pmid"]
    .count().sort_values(ascending=False).head(10).rename("pub_count")
)

## Section 3 — VOSviewer Corpus (Grant Abstracts)

In [ ]:
# Load and deduplicate grant data
df = pd.read_excel(OUTPUT_FILE, sheet_name="Grant Info", dtype=str)
print(f"Rows loaded: {len(df)}")

def core_grant_num(val):
    """Strip leading digit and trailing -XX suffix: 5T32AG012345-10 → T32AG012345."""
    if not isinstance(val, str):
        return val
    val = re.sub(r"^\d+", "", val.strip())
    val = re.sub(r"-\d+[A-Z]?$", "", val)
    return val

df["core_grant"] = df["project_num"].apply(core_grant_num)
df_unique = (
    df.sort_values("fiscal_year", ascending=True)
      .drop_duplicates(subset="core_grant", keep="last")
)

has_abstract = (
    df_unique["abstract_text"].dropna().astype(str).str.strip().str.len().gt(0).sum()
    if "abstract_text" in df_unique.columns else 0
)
print(f"Unique grants    : {len(df_unique)}")
print(f"With abstracts   : {has_abstract}")
print(f"Without abstracts: {len(df_unique) - has_abstract}")

In [ ]:
# Clean text and build corpus
# → output/vosviewer_input/vosviewer_corpus.txt
# → output/vosviewer_input/vosviewer_titles.txt

BOILERPLATE_HEADERS = [
    r"PROJECT SUMMARY/?ABSTRACT", r"PROJECT SUMMARY", r"ABSTRACT",
    r"DESCRIPTION\s*\(provided by applicant\)",
    r"DESCRIPTION\s*\(provided by the applicant\)",
    r"PUBLIC HEALTH RELEVANCE", r"RELEVANCE\s*\(See instructions\)",
    r"Narrative",
]

def clean_text(text) -> str:
    """Strip HTML, non-ASCII, NIH boilerplate headers, and extra whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    for pattern in BOILERPLATE_HEADERS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

corpus_lines = []
titles_lines = []
for _, row in df_unique.iterrows():
    title    = clean_text(row.get("project_title", ""))
    abstract = clean_text(row.get("abstract_text", ""))
    combined = f"{title}. {abstract}".strip(". ")
    if combined:
        corpus_lines.append(combined)
    if title:
        titles_lines.append(title)

CORPUS_FILE.write_text("\n".join(corpus_lines), encoding="utf-8")
TITLES_FILE.write_text("\n".join(titles_lines), encoding="utf-8")

print(f"Saved corpus : {CORPUS_FILE}  ({len(corpus_lines)} documents)")
print(f"Saved titles : {TITLES_FILE}  ({len(titles_lines)} documents)")

In [ ]:
# Save VOSviewer thesaurus / stopword file
# → output/vosviewer_input/vosviewer_thesaurus.txt
# Load in VOSviewer via the Thesaurus file option during map creation.

CUSTOM_STOPWORDS = [
    "addition", "advantage", "aim", "along", "also", "among",
    "area", "aspect", "available", "background", "basis", "broad",
    "career", "center", "challenge", "change", "combination", "commitment",
    "completion", "confer", "consistent", "continuation", "country", "current",
    "date", "decade", "department", "depth", "detail", "development",
    "director", "effort", "engagement", "environment", "evaluation",
    "excellence", "excellent", "experience", "experiential", "expertise",
    "facility", "field", "focus", "foundation", "fund", "future",
    "goal", "grant", "high", "highest", "home", "impart",
    "input", "institute", "integrated", "interface", "laboratory",
    "leadership", "leverage", "linkage", "major", "member", "mentor",
    "mentoring", "need", "new", "office", "order", "outstanding",
    "overall", "overarching", "perspective", "pipeline", "plan",
    "policy", "pool", "preparation", "primary", "priority", "program",
    "project", "provide", "range", "recruitment", "related", "relationship",
    "relevant", "report", "request", "research", "resource", "role",
    "school", "seminar", "significant", "solid", "specific", "strength",
    "strong", "structure", "substantive", "summary", "support", "synergy",
    "teaching", "term", "time", "total", "track", "train", "trainer",
    "training", "translation", "university", "use", "variety", "well", "work",
]

with open(THESAURUS_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["label", "replace by"])
    for word in sorted(CUSTOM_STOPWORDS):
        writer.writerow([word, ""])

print(f"Saved: {THESAURUS_FILE}  ({len(CUSTOM_STOPWORDS)} stopwords)")

## Section 4 — PubMed MEDLINE Export

Fetches full MEDLINE-format records for all publications.
Load the output file into VOSviewer via:
**Create → Bibliographic data → PubMed → select `grant_publications_medline.txt`**

In [ ]:
# Fetch MEDLINE records from PubMed
# → output/vosviewer_input/grant_publications_medline.txt
BATCH_SIZE = 200
DELAY      = 0.4

df_pubs_reload = pd.read_excel(OUTPUT_FILE, sheet_name="Publications", dtype=str)
print(f"Publications loaded: {len(df_pubs_reload):,}")

pmids_medline = (
    df_pubs_reload["pmid"]
    .dropna().astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .loc[lambda s: s != ""]
    .drop_duplicates().tolist()
)
print(f"Unique PMIDs to fetch: {len(pmids_medline):,}")


def fetch_medline_batch(pmid_batch: list[str]) -> str:
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed", "id": ",".join(pmid_batch),
        "rettype": "medline", "retmode": "text",
        "tool": "NIA_T32_LandscapeAnalysis", "email": EMAIL,
    }
    try:
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        return resp.text
    except Exception as e:
        print(f"  WARNING: {e}")
        return ""


batches = [pmids_medline[i:i+BATCH_SIZE] for i in range(0, len(pmids_medline), BATCH_SIZE)]
total_fetched = 0

with open(MEDLINE_FILE, "w", encoding="utf-8") as f:
    for i, batch in enumerate(batches):
        print(f"  Batch {i+1}/{len(batches)}  ({total_fetched:,} records written)...")
        text = fetch_medline_batch(batch)
        if text:
            f.write(text)
            total_fetched += text.count("\nPMID-") + (1 if text.startswith("PMID-") else 0)
        time.sleep(DELAY)

print(f"\nDone! {total_fetched:,} records saved to {MEDLINE_FILE}")

## Section 5 — VOSviewer Map Analysis

---
### ⚠️ CLAUDE-ASSISTED STEP — First run

**Before running the cells in this section, you need to:**

1. Run VOSviewer on `output/vosviewer_input/grant_publications_medline.txt`
2. Export the VOSviewer **map file** and **network file** (File → Export) and save both to `output/vosviewer_output/`
3. Save any VOSviewer screenshot images to `output/vosviewer_images/`
4. **Share the map file with Claude** and use this prompt:

> *"Here is my new VOSviewer map file. Please identify the measurement and psychometrics terms, flag any false positives, name each cluster, and produce an updated `measurement_psychometric_terms_v2.csv` and `false_positive_terms.csv`. Also give me an updated `cluster_names` dictionary to paste into my notebook."*

5. Save both CSVs Claude returns to `output/measurement_analysis/`
6. Paste the updated `cluster_names` dictionary into the settings cell below

---
### ⚠️ CLAUDE-ASSISTED STEP — Re-runs with new publications

If you re-ran VOSviewer with additional publications, share both the **new map file** and the **existing `measurement_psychometric_terms_v2.csv`** with Claude and use this prompt:

> *"Here is my updated VOSviewer map file and my existing measurement terms CSV. Please check for new measurement-related terms not in the previous run, flag any that changed clusters, and produce an updated `measurement_psychometric_terms_v2.csv`. Also give me an updated `cluster_names` dictionary."*

---

In [ ]:
# ── USER SETTINGS for Section 5 ───────────────────────────────────────────────
# Update these filenames if your VOSviewer exports have different names
MAP_FILE  = DIR_VOS_OUTPUT / "vosviewer_map_file_pubs.txt"
MEAS_FILE = DIR_MEAS / "measurement_psychometric_terms_v2.csv"
FP_FILE   = DIR_MEAS / "false_positive_terms.csv"
# ─────────────────────────────────────────────────────────────────────────────

# ── CLUSTER NAMES — paste updated dict from Claude after each VOSviewer run ───
# VOSviewer may renumber clusters on each run — always get fresh names from Claude
cluster_names = {
    1:  "Clinical care & caregiving (catch-all)",
    2:  "Physical function, fatigue & mobility",
    3:  "Neuroinflammation & mouse models",
    4:  "Cell biology & molecular signaling",
    5:  "Neurodegeneration & brain imaging",
    6:  "Alzheimer's & protein pathology",
    7:  "Bone density, genetics & body composition",
    8:  "Skeletal muscle & metabolic physiology",
    9:  "Thermoregulation & cardiovascular physiology",
    10: "Epigenetics, TDP-43 & COVID",
    11: "Cell signaling & inhibition pathways",
    12: "Mitochondria & oxidative biology",
    13: "Cancer, imaging & clinical coding",
    14: "Knee osteoarthritis & pain",
    15: "Gene expression & stem cell biology",
    16: "Neurological disorders & infectious disease",
    17: "Endocrinology & diet",
    18: "Immunology & tumor biology",
    19: "Genetics & hematopoiesis",
    20: "Ophthalmology & glaucoma",
    21: "Cell death pathways",
    22: "Hormones & competition",
    23: "Miscellaneous", 24: "Miscellaneous", 25: "Miscellaneous",
}
# ─────────────────────────────────────────────────────────────────────────────

# Check that required files exist before proceeding
missing = [f for f in [MAP_FILE, MEAS_FILE] if not f.exists()]
if missing:
    print("ERROR — the following required files are missing:")
    for f in missing:
        print(f"  {f}")
    print("\nPlease follow the Claude-assisted steps above before running this cell.")
else:
    df_map  = pd.read_csv(MAP_FILE, sep="\t")
    df_map['label'] = df_map['label'].astype(str)
    df_map["cluster_name"] = df_map["cluster"].map(cluster_names)
    meas_df = pd.read_csv(MEAS_FILE)
    print(f"Map file loaded  : {len(df_map):,} terms, {df_map['cluster'].nunique()} clusters")
    print(f"Measurement terms: {len(meas_df)}")

In [ ]:
# Build shared cluster color map (used by Charts 3 and 4)
unique_clusters   = sorted(meas_df["cluster"].unique())
palette           = plt.colormaps["tab10"].resampled(len(unique_clusters))
cluster_color_map = {c: palette(i) for i, c in enumerate(unique_clusters)}
print(f"Color map built for {len(unique_clusters)} clusters.")

In [ ]:
# ── CHART 1: Dominant Research Clusters
# → output/charts/chart_dominant_clusters.png

cluster_totals = (
    df_map.groupby(["cluster", "cluster_name"])["weight<Occurrences>"]
    .sum().reset_index()
    .sort_values("weight<Occurrences>", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    cluster_totals["cluster_name"][::-1],
    cluster_totals["weight<Occurrences>"][::-1],
    color="#2a78d6"
)
ax.set_xlabel("Total occurrences across publications")
ax.set_title("Dominant Research Clusters in NIA T32 Publication Corpus",
             fontsize=13, fontweight="bold")
ax.bar_label(bars, padding=3, fmt="%.0f")
plt.tight_layout()
out = DIR_CHARTS / "chart_dominant_clusters.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

In [ ]:
# ── CHART 2: Measurement & Psychometrics Terms (top 20)
# → output/charts/chart_measurement_terms.png

top_meas = meas_df.nlargest(20, "occurrences")

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    top_meas["term"][::-1],
    top_meas["occurrences"][::-1],
    color="#eda100"
)
ax.set_xlabel("Occurrences across publications")
ax.set_title("Measurement & Psychometrics Terms in NIA T32 Publication Corpus",
             fontsize=13, fontweight="bold")
ax.bar_label(bars, padding=3, fmt="%.0f")
plt.tight_layout()
out = DIR_CHARTS / "chart_measurement_terms.png"
plt.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

In [ ]:
# ── CHART 3: Spatial scatter of measurement terms on the VOSviewer map
# → output/charts/chart_measurement_terms_spatial.png

map_coords   = df_map[["label", "x", "y"]].rename(columns={"label": "term"})
meas_spatial = meas_df.merge(map_coords, on="term", how="left")

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(df_map["x"], df_map["y"], s=8, color="lightgray", alpha=0.4, zorder=1)

for c in unique_clusters:
    subset = meas_spatial[meas_spatial["cluster"] == c]
    ax.scatter(
        subset["x"], subset["y"],
        s=subset["occurrences"] * 8,
        color=cluster_color_map[c],
        edgecolor="white", linewidth=0.5,
        label=f"Cluster {c}: {cluster_names.get(c, 'Unknown')}",
        zorder=3
    )

for _, row in meas_spatial.nlargest(10, "occurrences").iterrows():
    ax.annotate(row["term"], (row["x"], row["y"]),
                fontsize=7, xytext=(4, 4), textcoords="offset points")

ax.set_title(
    "Measurement & Psychometrics Terms Are Spatially Scattered\n(dot size = occurrences, color = cluster)",
    fontsize=13, fontweight="bold")
ax.set_xlabel("VOSviewer map dimension 1")
ax.set_ylabel("VOSviewer map dimension 2")
ax.legend(loc="upper left", fontsize=7, framealpha=0.9,
          bbox_to_anchor=(1.02, 1), borderaxespad=0)
plt.tight_layout()
out = DIR_CHARTS / "chart_measurement_terms_spatial.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

In [ ]:
# ── CHART 4: All measurement terms colored by cluster (horizontal bar)
# → output/charts/chart_measurement_terms_by_cluster.png

meas_by_cluster = meas_df.sort_values(["cluster", "occurrences"], ascending=[True, False])
bar_colors = meas_by_cluster["cluster"].map(cluster_color_map)

fig, ax = plt.subplots(figsize=(10, 14))
bars = ax.barh(
    meas_by_cluster["term"],
    meas_by_cluster["occurrences"],
    color=bar_colors
)
ax.set_xlabel("Occurrences across publications")
ax.set_title(
    "Measurement & Psychometrics Terms by Cluster\n(color = cluster membership)",
    fontsize=13, fontweight="bold")
ax.bar_label(bars, padding=3, fmt="%.0f", fontsize=8)
ax.invert_yaxis()

legend_handles = [plt.Rectangle((0,0),1,1, color=cluster_color_map[c]) for c in unique_clusters]
legend_labels  = [f"Cluster {c}: {cluster_names.get(c, 'Unknown')}" for c in unique_clusters]
ax.legend(legend_handles, legend_labels, loc="lower right", fontsize=8, framealpha=0.9)

plt.tight_layout()
out = DIR_CHARTS / "chart_measurement_terms_by_cluster.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────

total_occ = df_map["weight<Occurrences>"].sum()
meas_occ  = meas_df["occurrences"].sum()

print(f"Total terms in map          : {len(df_map):,}")
print(f"Total occurrences           : {total_occ:,}")
print(f"Measurement term occurrences: {meas_occ:,}  ({meas_occ/total_occ*100:.2f}% of total)")
print(f"Unique measurement terms    : {len(meas_df)}")

print("\nMeasurement terms by cluster:")
display(
    meas_df.groupby(["cluster", "cluster_name"])["occurrences"]
    .agg(n_terms="count", total_occ="sum")
    .sort_values("total_occ", ascending=False)
)

In [ ]:
# ── Final folder listing — confirm all files are in the right places ───────────

print(f"Output folder contents:\n")
for f in sorted(OUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1_000_000:.1f} MB"
        print(f"  {str(f.relative_to(OUT_DIR)):<65}  {size_str}")